# Halide Perovskite — Single Trial Pre-Flight Check

Submits **one acquisition trial** through the full queue-server stack so you
can verify the end-to-end path (pumps → QEPro → evaluation) before committing
to a full campaign run.

## ⚠️ Physical safety — read before running

> **Disconnect the DDS and Ultra syringes from their drive screws before
> running this notebook.**  The pump motors will receive a move command and
> the screws will turn.  Without disconnection you will dispense fluid.

## Prerequisites

1. `validate_paths.ipynb` ran cleanly — all devices connected.

2. Queue-server worker running with the halide profile loaded:
   ```bash
   XPD_HALIDE_CONFIG=/path/to/configs/halide_beamline.json \\
   XPD_HTTP_API_KEY=$QSERVER_HTTP_SERVER_API_KEY \\
       start-re-manager --startup-module xpd_tools.optimization.profiles.halide
   ```

3. A saved campaign config JSON produced by `halide_beamline_updated.ipynb`
   (e.g. `configs/halide_beamline.json`).  Set `XPD_HALIDE_CONFIG` in your
   `.env` or shell to point at it.

## Run this notebook

```bash
cd ~/Documents/AutonomousExperiments/HalideExchange/uvvis
jupyter notebook single_trial.ipynb
```

## Beamline bring-up sequence

| Step | Notebook / command |
|------|-------------------|
| 1 | `validate_paths.ipynb` — device & PV connectivity (read-only) |
| 2 | `single_trial.ipynb` — **this notebook** — one acquisition trial |
| 3 | `halide_beamline_updated.ipynb` — full optimization campaign |


---
## 1 — Environment

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

env_keys = [
    "QSERVER_HTTP_URI",
    "QSERVER_HTTP_SERVER_API_KEY",
    "ZMQ_CONSUMER_ADDR",
    "TILED_URI",
    "XPD_HALIDE_CONFIG",
]

print("Environment variables:")
print("-" * 50)
for key in env_keys:
    val = os.environ.get(key, "<not set>")
    if "KEY" in key and val != "<not set>":
        val = val[:6] + "..." if len(val) > 6 else "***"
    print(f"  {key:<35} {val}")

CONFIG_PATH: str = os.environ.get("XPD_HALIDE_CONFIG", "configs/halide_beamline.json")
print()
print(f"  Config path : {CONFIG_PATH}")
print(f"  Config exists: {Path(CONFIG_PATH).exists()}")

---
## 2 — Load campaign config

Reconstructs the `BuildAgent` from the archived JSON so the single trial
uses exactly the same experiment parameters as the campaign.

In [ ]:
from xpd_tools.optimization.campaign import load_campaign_config

build_agent = load_campaign_config(CONFIG_PATH)

print("BuildAgent reconstructed:")
print(f"  evaluation_method : {build_agent.evaluation_method}")
print(f"  DOFs              : {[p.id for p in build_agent.pumps]}")
print(f"  run.iterations    : {build_agent.run_settings.iterations}")

---
## 3 — Build agent and verify plan registration

In [ ]:
from xpd_tools.optimization.campaign import assert_plan_registered

run_agent = build_agent.build()   # -> QueueserverAgent

print("QueueserverAgent built.")
print(f"  acquisition_plan : {run_agent.acquisition_plan}")
print(f"  actuators        : {list(run_agent.actuators)}")
print()

assert_plan_registered(run_agent, plan_name="uvvis_acquire")

---
## 4 — Define trial point

Adjust rates (ul/min) to match your current reagent setup.
Every rate must fall within its pump's min/max range for the loaded syringe.

| DOF key | Device | Reagent |
|---------|--------|---------|
| `infusion_rate_CsPb` | `dds2_p1` | CsPb precursor |
| `infusion_rate_Br`   | `dds2_p2` | TOABr |
| `infusion_rate_Cl`   | `dds3_p1` | Pb-rich / TOACl |
| `infusion_rate_I`    | `dds3_p2` | TOAI |

In [ ]:
# Rates in ul/min — adjust to your reagent setup.
single_trial_point: list[dict[str, float]] = [{
    "infusion_rate_CsPb": 40.0,   # dds2_p1  -- watch this screw
    "infusion_rate_Br":   30.0,   # dds2_p2
    "infusion_rate_Cl":   10.0,   # dds3_p1
    "infusion_rate_I":    20.0,   # dds3_p2
}]

print("Trial point:")
for k, v in single_trial_point[0].items():
    print(f"  {k:<25} {v} ul/min")

---
## 5 — Submit and block

`submit_suggestions` sends the point to the queue server and returns a
`Future`.  `.result()` blocks until the acquisition and evaluation are done.

**What to watch:**
- Drive screws begin turning within a few seconds of submission.
- Console output from the worker shows dark/reference/sample spectrum capture.
- A failed peak fit on a dark frame is expected and useful — it means the
  acquisition ran but found no PL signal (correct for a dark frame).

In [ ]:
print("Submitting trial to queue server...")
future = run_agent.submit_suggestions(single_trial_point)

print("Waiting for acquisition and evaluation to complete...")
result = future.result()   # blocks until done

print("\nResult:")
print(result)

---
## Next Steps

| Outcome | Next action |
|---------|-------------|
| Screws turned, result returned | Reconnect syringes and open `halide_beamline_updated.ipynb` |
| Screws did not turn | Check pump EPICS IOC, DDS comms state (`Enabled`), and pump rate bounds |
| Evaluation error | Inspect worker logs; a dark-frame fit failure is expected and harmless |
